# Varaha PPO Training Demo

This notebook demonstrates RL training for the Varaha wildfire logistics drone simulator.

**Pipeline:**
1. Install dependencies
2. Run PPO training (Stable-Baselines3) on hard environments
3. Evaluate trained policy and show reward curve

**Modes:**
- **Quick demo** (~15 min): `quick_15m=True` — reduced timesteps/envs for fast iteration
- **Full training** (~90 min): `quick_15m=False` — production run on H100

In [ ]:
# Setup: ensure we're in project root
# Uncomment the path that matches your environment:
# %cd /root/project
# %cd /content/varaha  # Colab
%cd /root/project

In [ ]:
!pip -q install stable-baselines3 gymnasium matplotlib numpy torch

## Run Training

The training script (`train_ppo.py`) trains a PPO agent on randomized hard worlds with obstacles, hazards, responders, and dynamic events.

In [ ]:
# Option A: Run via CLI (output in terminal)
# !python train_ppo.py --quick-15m --device cuda --vec-backend dummy --save-dir ./results_training_demo

In [ ]:
# Option B: Run training in-notebook (output appears below) — recommended for judges
import sys
sys.path.insert(0, '.')
from train_ppo import train

# Quick demo (~15 min): reduced timesteps, smaller net, fewer envs
# Set quick_15m=False for full 90-min run
model = train(
    total_timesteps=12_000_000,
    n_envs=24,
    save_dir='./results_training_demo',
    time_limit_s=900,
    v2=False,
    hard_mix=False,
    vec_backend='dummy',  # use 'subproc' for faster parallel envs
    device='cuda',
    quick_15m=True,
    eval_episodes=8,
    trace_count=2,
)
print('Training complete. Model saved to results_training_demo/')

## Show Results

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

metrics_path = Path('results_training_demo/metrics_hardcore.json')
if not metrics_path.exists():
    print('Run training first. Metrics not found at', metrics_path)
else:
    with metrics_path.open() as f:
        m = json.load(f)

    print('=== Training Results ===')
    print(f"Timesteps:     {m.get('training_timesteps', 0):,}")
    print(f"Training time: {m.get('training_seconds', 0)/60:.1f} min")
    print(f"Mean reward:   {m.get('mean_reward', 0):.1f} ± {m.get('std_reward', 0):.1f}")
    print(f"Success rate:  {m.get('success_rate', 0):.1%}")
    print(f"Deliveries:    {m.get('mean_deliveries', 0):.2f}")

In [ ]:
# Plot reward curve if available
from pathlib import Path
import matplotlib.pyplot as plt

curve_path = Path('results_training_demo/reward_curve_hardcore.png')
if curve_path.exists():
    from IPython.display import Image, display
    display(Image(filename=str(curve_path), width=700))
else:
    print('Reward curve not found. Run training first.')